In [ ]:
# Import the libraries required for environment variables, JSON handling, Markdown display, and OpenAI.
import os
import json

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
# Load the OpenAI API key from the environment and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")

openai = OpenAI()


In [ ]:
# Define a sample email that will be used to test the intelligence pipeline.
email = """
Subject: Meeting About the New AI Automation Project

Hi John,

I wanted to update you about the AI automation project we discussed.

The development team has completed the initial workflow and the testing phase will begin next Monday.

We need you to review the workflow before Friday and provide any feedback.

We will have a project meeting on Tuesday at 10:00 AM to discuss the testing results and determine the next steps.

Please also send the team the final list of automation requirements before the meeting.

Best,
David
"""


In [ ]:
# Define the instructions that control email classification and complete intelligence extraction.
intelligence_prompt = """
You are an email intelligence assistant.

Analyze the email and return valid JSON using exactly this structure:

{
    "summary": "",
    "category": "",
    "action_items": [],
    "deadlines": [],
    "meetings": [],
    "people": [],
    "questions": [],
    "decisions": []
}

The category must be exactly one of:
- billing
- technical
- account
- sales
- general

Category rules:
- billing: Payments, charges, invoices, refunds, or subscription billing issues.
- technical: Bugs, errors, APIs, integrations, or software problems.
- account: Login, password, account access, or account settings.
- sales: Clear buying intent, purchase requests, demos, or enterprise sales inquiries.
- general: Information requests without clear buying intent.

A question about pricing, features, or plans is NOT automatically sales.
Only classify as sales when there is clear buying intent.

For meetings, return objects containing:
- subject
- date
- time

Only extract information explicitly present in the email.
Do not invent or assume information.
"""


In [ ]:
# Define the required fields and create a validator for the LLM's structured response.
intelligence_fields = [
    "summary",
    "category",
    "action_items",
    "deadlines",
    "meetings",
    "people",
    "questions",
    "decisions"
]


def validate_intelligence(data):
    if not data:
        return False

    return all(field in data for field in intelligence_fields)


In [ ]:
# Analyze an email with one LLM call, parse the JSON response, validate it, and return the result.
def analyze_email(email):
    messages = [
        {"role": "system", "content": intelligence_prompt},
        {"role": "user", "content": email}
    ]

    try:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )

        result = response.choices[0].message.content
        data = json.loads(result)

        if not validate_intelligence(data):
            print("The LLM response is missing required fields.")
            return None

        return data

    except json.JSONDecodeError:
        print("The LLM returned invalid JSON.")
        return None

    except Exception as error:
        print(f"An error occurred: {error}")
        return None


In [ ]:
# Run the complete email intelligence pipeline and store the structured result.
result = analyze_email(email)

print(result)


In [ ]:
# Display each part of the email intelligence result in a readable format.
for field, value in result.items():
    print(f"\n{field.upper()}:")
    print(value)


In [ ]:
# Create a small evaluation dataset to test whether the classifier handles different email categories.
evaluation_emails = [
    {
        "expected": "billing",
        "email": "I noticed that my card was charged twice for the same subscription. Please help me resolve this."
    },
    {
        "expected": "account",
        "email": "I forgot my password and cannot access my account. How can I reset it?"
    },
    {
        "expected": "technical",
        "email": "The API returns a 500 error whenever I try to upload a document."
    },
    {
        "expected": "sales",
        "email": "Our company is interested in purchasing the enterprise plan. Can we schedule a demo?"
    },
    {
        "expected": "general",
        "email": "What features are available in your basic plan?"
    },
    {
        "expected": "billing",
        "email": "Can you explain why my latest invoice is higher than the previous one?"
    },
    {
        "expected": "technical",
        "email": "The integration stopped working after we connected our CRM."
    },
    {
        "expected": "account",
        "email": "I need to change the email address associated with my account."
    }
]


In [ ]:
# Evaluate the classifier across the test dataset and calculate its classification accuracy.
correct = 0

for item in evaluation_emails:
    result = analyze_email(item["email"])
    predicted = result["category"]

    print(f"Expected: {item['expected']}")
    print(f"Predicted: {predicted}")
    print("-" * 40)

    if predicted == item["expected"]:
        correct += 1

accuracy = correct / len(evaluation_emails)

print(f"Accuracy: {accuracy:.2%}")
